In [ ]:
# Code for BirdCLEF 2026 competition

import librosa
import numpy as np
import pandas as pd
import os

In [ ]:
# train_soundscapes_labels has the following information for each file in train_soundscapes:
# - The filename of the soundscape
# - The primary label (the bird species that is most prominently featured in the soundscape)
# - The time intervals during which the primary label is present in the soundscape

# For each broad class of animals, we classify the soundscapes that primarily feature that class.

taxonomy_df = pd.read_csv("taxonomy.csv")

# The problem with this code is that each 5 second interval should be treated as a separate data point.

train_soundscapes_labels_df = pd.read_csv("train_soundscapes_labels.csv")

# train_soundscapes_reduced_df treats every 5 second interval as a separate data point.

train_soundscapes_reduced_df = pd.DataFrame({
    "filename": train_soundscapes_labels_df.iloc[:, :3].astype(str).agg(''.join, axis=1),
    "primary_label": train_soundscapes_labels_df.iloc[:, 3]
})

In [ ]:
# For each file, we list all animals that are present in the file
# We then classify the file as belonging to the class of each of those animals.

classes = ['Insecta', 'Reptilia', 'Amphibia', 'Mammalia', 'Aves']
filenames = {value: [] for value in classes}

for entry in train_soundscapes_reduced_df.itertuples():
    filename = entry.filename
    primary_label = entry.primary_label.split(';') # This records all species in a file.

    found = {value: False for value in classes}
    
    for label in primary_label:
        for value in classes:
            if taxonomy_df[taxonomy_df['primary_label'] == label]['class_name'].values[0] == value:
                found[value] = True
    
    for value in classes:
         if found[value]:
            filenames[value].append(filename)

In [ ]:
# Number of files in each class
# It is troubling that only 26 of the files have reptiles.

print(f"Insecta: {len(filenames['Insecta'])}")
print(f"Reptilia: {len(filenames['Reptilia'])}")
print(f"Amphibia: {len(filenames['Amphibia'])}")
print(f"Mammalia: {len(filenames['Mammalia'])}")
print(f"Aves: {len(filenames['Aves'])}")

Insecta: 336
Reptilia: 26
Amphibia: 1132
Mammalia: 82
Aves: 424


In [ ]:
# For each class, creates a list of non-elements of that class.
# We cannot include all non-elements of a class, because there are too many.
# Instead, we will make the list as long as the list of elements of that class.
# Also shuffles the elements of each class.

import random

filenames_non_elements = {value : [] for value in classes}
filenames_all = list(set().union(*filenames.values()))

# The following loop works for every class except Amphibia because that class has too many elements.
# We add 80% of the elements of each class to the training set, and 20% to the test set.
# This loop chooses random elements to add to the class because the data tends to be ordered by location.
# We also don't want too many 5 second intervals from the same 1 minute audio clip to be in the same set.

for value in ['Insecta', 'Reptilia', 'Mammalia', 'Aves']:
    length_current = 0
    length_target = len(filenames[value])
    random.shuffle(filenames[value])
    while length_current < length_target:
        filename = random.choice(filenames_all)
        if filename not in filenames[value] and filename not in filenames_non_elements[value]:
            filenames_non_elements[value].append(filename)
            length_current += 1

# This code lists all filenames which do not belong to Amphibia.

filenames_non_elements['Amphibia'] = [filename for filename in filenames_all if filename not in filenames['Amphibia']]

length = len(filenames_non_elements['Amphibia'])

# This line truncates the elements of Amphibia to be the same length as the non-elements of Amphibia.

random.shuffle(filenames['Amphibia'])
filenames['Amphibia'] = filenames['Amphibia'][:length]


In [ ]:
# Creates train and test sets for each class, with an 80-20 split.
# There is also a train-test split for non-elements of each class.

filenames_train = {value: filenames[value][:int(0.8 * len(filenames[value]))] for value in classes}
filenames_test = {value: filenames[value][int(0.8 * len(filenames[value])):] for value in classes}
filenames_non_elements_train = {value: filenames_non_elements[value][:int(0.8 * len(filenames_non_elements[value]))] for value in classes}
filenames_non_elements_test = {value: filenames_non_elements[value][int(0.8 * len(filenames_non_elements[value])):] for value in classes}

In [ ]:
# The previous window justs lists the filenames for each train and test set.
# This code actually finds the audio data corresponding to each filename.

train_dfs = {value: [] for value in classes}
test_dfs = {value: [] for value in classes}
train_dfs_non_elements = {value: [] for value in classes}
test_dfs_non_elements = {value: [] for value in classes}

for value in classes:
    for filename in filenames_train[value]:
        file = filename[:-16]
        file_start = filename[-16:-8]
        file_end = filename[-8:]
        audio, sample_rate = librosa.load(f"train_soundscapes/{file}", sr=None)
        train_dfs[value].append((audio, sample_rate, file_start, file_end))
    
    for filename in filenames_test[value]:
        file = filename[:-16]
        file_start = filename[-16:-8]
        file_end = filename[-8:]
        audio, sample_rate = librosa.load(f"train_soundscapes/{file}", sr=None)
        test_dfs[value].append((audio, sample_rate, file_start, file_end))
    
    for filename in filenames_non_elements_train[value]:
        file = filename[:-16]
        file_start = filename[-16:-8]
        file_end = filename[-8:]
        audio, sample_rate = librosa.load(f"train_soundscapes/{file}", sr=None)
        train_dfs_non_elements[value].append((audio, sample_rate, file_start, file_end))
    
    for filename in filenames_non_elements_test[value]:
        file = filename[:-16]
        file_start = filename[-16:-8]
        file_end = filename[-8:]
        audio, sample_rate = librosa.load(f"train_soundscapes/{file}", sr=None)
        test_dfs_non_elements[value].append((audio, sample_rate, file_start, file_end))

In [ ]:
# This code uses a logistic regression model with PCA for dimensionality reduction for each class.
# It creates a separate classifier for each class.

# The code really struggled to converge for the Aves class.
# I had to increase the number of iterations to 500 to get it to converge.
# Even then, it took a long time to run.
# Interestingly, the code converged much faster for Amphibia, which has more data points than Aves.

# The model also wrote:
# ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
#STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

#Increase the number of iterations to improve the convergence (max_iter=500).
#You might also want to scale the data as shown in:
#    https://scikit-learn.org/stable/modules/preprocessing.html
#Please also refer to the documentation for alternative solver options:
#    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
#  n_iter_i = _check_optimize_result(

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from time import time

models = {}
X_test_pca_dict = {}
y_test_combined_dict = {}


for value in classes:
    t = time()
    print(f"Processing class {value}...")
    combined_train = [(df[0], 1) for df in train_dfs[value]] + [(df[0], 0) for df in train_dfs_non_elements[value]]
    combined_test = [(df[0], 1) for df in test_dfs[value]] + [(df[0], 0) for df in test_dfs_non_elements[value]]

    X_train_combined = [audio for audio, _ in combined_train]
    X_test_combined = [audio for audio, _ in combined_test]

    y_train_combined = [label for _, label in combined_train]
    y_test_combined_dict[value] = [label for _, label in combined_test]

# StandardScaler converts all of the data to a standard Normal distribution.

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_combined)
    X_test_scaled = scaler.transform(X_test_combined)

# PCA dramatically reduces the number of components in each graph.

    pca = PCA(n_components=min(100, len(X_train_scaled)))

    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca_dict[value] = pca.transform(X_test_scaled)

    model = LogisticRegression(max_iter=500)
    model.fit(X_train_pca, y_train_combined)
    models[value] = model
    print(f"Time taken for class {value}: {time() - t:.2f} seconds")


Processing class Insecta...
Time taken for class Insecta: 353.63 seconds
Processing class Reptilia...
Time taken for class Reptilia: 7.58 seconds
Processing class Amphibia...
Time taken for class Amphibia: 49.83 seconds
Processing class Mammalia...
Time taken for class Mammalia: 32.40 seconds
Processing class Aves...
Time taken for class Aves: 735.31 seconds


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
# Running times for each class:
# Insecta: 353.63 seconds (336)
# Reptilia: 7.58 seconds (26)
# Amphibia: 49.83 seconds (1132)
# Mammalia: 32.40 seconds (82)
# Aves: 735.31 seconds (424)

# This code evaluates the model on the test set and prints the accuracy. Perfect score!

# To run this, I would have to keep track of X_test_pca and y_test_combined for each class.

for value in classes:
    model = models[value]
    y_pred = model.predict(X_test_pca_dict[value])
    accuracy = accuracy_score(y_test_combined_dict[value], y_pred)
    print(f"Accuracy for {value}: {accuracy:.2f}")

Accuracy for Insecta: 1.00
Accuracy for Reptilia: 0.92
Accuracy for Amphibia: 0.99
Accuracy for Mammalia: 0.97
Accuracy for Aves: 0.93


In [ ]:
# Code copied from ChatGPT to build a CNN model for binary classification of spectrograms.
# Turns audio file into a spectrogram, and then uses a CNN to classify the spectrogram.
# I have not seriously looked into this yet, but it seems promising.

import tensorflow as tf
from tensorflow.keras import layers, models

def build_model(input_shape=(128, 313, 1)):
    model = models.Sequential()

    # Block 1
    model.add(layers.Conv2D(16, (3, 3), activation='relu', padding='same', input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Block 2
    model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Block 3
    model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Global pooling instead of Flatten (reduces overfitting)
    model.add(layers.GlobalAveragePooling2D())

    # Dense head
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(0.4))

    model.add(layers.Dense(1, activation='sigmoid'))  # binary classification

    return model